Copyright (C) 2026  Diarmuid

This program is free software: you can redistribute it and/or modify\
it under the terms of the GNU General Public License as published by\
the Free Software Foundation, either version 3 of the License, or\
(at your option) any later version.

This program is distributed in the hope that it will be useful,\
but WITHOUT ANY WARRANTY; without even the implied warranty of\
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the\
GNU General Public License for more details.

You should have received a copy of the GNU General Public License\
along with this program.  If not, see <https://www.gnu.org/licenses/>.

----------------------------------------------------------------------

Author: Diarmuid O'Neill\
Student Number: C00282898\
Date: 10/02/2026\
Brief Description:
This notebook aims to use a Linear SVM to predict whether a particular sonar response coresponds to a rock or a metal cylinder (a sea mine). Each datapoint consists of 60 numbers in the range 0.0 to 1.0 where each number represents the energy within a particular frequency band, integrated over a certain period of time. \
Source: [UCI Machine Learning Repository - Sonar, Mines vs Rocks](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)

In [15]:
# Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score, classification_report

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Importing Dataset
dataset = pd.read_csv("sonar.all-data", header=None)
dataset.head()

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


First the correlation coefficients are examined to see which features might be good predictors of the target class. The target is a binary class consisting of "R" for rock or "M" for mine. The correlation coefficients below are in descending order.

In [3]:
last_column = dataset.columns[-1]
dataset[last_column] = dataset[last_column].map({"R":0, "M":1})
dataset = dataset.apply(pd.to_numeric, errors="coerce")
pd.set_option('display.max_rows', None)
correlation_matrix = dataset.corr()

print(correlation_matrix[last_column].sort_values(ascending=False))

60    1.000000
10    0.432855
11    0.392245
48    0.351312
9     0.341142
44    0.339406
47    0.329333
8     0.321448
12    0.312811
45    0.305610
46    0.301697
50    0.293205
51    0.288651
43    0.273779
0     0.271694
3     0.250638
20    0.242755
42    0.234077
1     0.231238
19    0.224115
4     0.222232
21    0.201361
2     0.192195
7     0.189314
57    0.184191
53    0.182687
49    0.180224
13    0.157277
41    0.144147
52    0.141871
18    0.138204
5     0.132327
58    0.130826
55    0.129341
22    0.127413
6     0.114748
54    0.095639
59    0.090055
27    0.082486
23    0.074041
14    0.057318
38    0.056910
26    0.054997
28    0.036639
25    0.028063
24    0.025306
40    0.020942
17    0.014612
15    0.011581
29    0.004219
56    0.000933
16   -0.003969
39   -0.035928
37   -0.040527
31   -0.055275
32   -0.107184
30   -0.110728
33   -0.172010
36   -0.209055
34   -0.227670
35   -0.269151
Name: 60, dtype: float64


The code block below takes the two features with the highest correlation to the target variable (columns 10 and 11) and splits the dataset into training, validation and test sets. Note that there are only 209 data points in total so these splits will be rather small and may lead to some variability in results. It also trains the model and runs it against the validation set to get a baseline validation accuracy and classification report.

In [4]:
X = dataset[[10, 11]]
X = X.values

y = dataset[last_column]
y = y.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=12345
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=12345
)

svm_model = LinearSVC(random_state=12345)
svm_model.fit(X_train, y_train)

y_val_pred = svm_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy: ", val_accuracy)

print(classification_report(y_val, y_val_pred))

Validation Accuracy:  0.7619047619047619
              precision    recall  f1-score   support

           0       0.80      0.73      0.76        22
           1       0.73      0.80      0.76        20

    accuracy                           0.76        42
   macro avg       0.76      0.76      0.76        42
weighted avg       0.77      0.76      0.76        42



The classification report above shows an initial validation accuracy of 76%. It also shows precision results for each class (80% for rocks, 73% for mines). This means that for every time the model predicts a rock, how often was it correct and likewise for mines. Recall is particularly important for this task because it tells us, of all the rocks/mines how many were correctly predicted. This is important for mines because in a real world scenario a missed mine is far worse than a false alarm. Recall for mines in the validation set currently stands at 80% and for rocks 73%.

In [5]:
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Test Accuracy: 0.7380952380952381
              precision    recall  f1-score   support

           0       0.76      0.65      0.70        20
           1       0.72      0.82      0.77        22

    accuracy                           0.74        42
   macro avg       0.74      0.73      0.73        42
weighted avg       0.74      0.74      0.74        42



Test accuracy is slightly lower at 74%. Precision for rocks is also slightly lower at 76%. Precision for mines is 72%. Recall for rocks is 65% and slightly higher for mines at 82%. It is worth noting that the variability between the test and validation set results could be down to the small sample sizes. The next code block performs the same SVM classification but this time uses the top four most correlated features with the target.

In [6]:
X2 = dataset[[10, 11, 48, 9]]
X2 = X2.values

y2 = dataset[last_column]
y2 = y2.values

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=12345
)

X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_train2, y_train2, test_size=0.25, random_state=12345
)

svm_model2 = LinearSVC(random_state=12345)
svm_model2.fit(X_train2, y_train2)

y_val_pred2 = svm_model2.predict(X_val2)
val_accuracy2 = accuracy_score(y_val2, y_val_pred2)
print("Validation Accuracy: ", val_accuracy2)

print(classification_report(y_val2, y_val_pred2))

Validation Accuracy:  0.6904761904761905
              precision    recall  f1-score   support

           0       0.71      0.68      0.70        22
           1       0.67      0.70      0.68        20

    accuracy                           0.69        42
   macro avg       0.69      0.69      0.69        42
weighted avg       0.69      0.69      0.69        42



In [7]:
svm_model2 = LinearSVC(random_state=12345)
svm_model2.fit(X_train2, y_train2)
y_pred2 = svm_model2.predict(X_test2)

accuracy2 = accuracy_score(y_test2, y_pred2)
print("Test Accuracy:", accuracy2)

print(classification_report(y_test2, y_pred2))

Test Accuracy: 0.8095238095238095
              precision    recall  f1-score   support

           0       0.83      0.75      0.79        20
           1       0.79      0.86      0.83        22

    accuracy                           0.81        42
   macro avg       0.81      0.81      0.81        42
weighted avg       0.81      0.81      0.81        42



When using the top four most correlated features with the target we can see a somewhat lowered validation accuracy but a marked improvement in test accuracy. Test accuracy jumps from 74% to 81%. This shows that careful feature selection appears to help improve accuracy. There are also noticeable improvements in both precision and recall. The next code block tries out the same SVM classification model, this time using all the available features.

In [8]:
X3 = dataset.iloc[:, :-1]
X3 = X3.values

y3 = dataset[last_column]
y3 = y3.values

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=12345
)

X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_train3, y_train3, test_size=0.25, random_state=12345
)

svm_model3 = LinearSVC(random_state=12345)
svm_model3.fit(X_train3, y_train3)

y_val_pred3 = svm_model3.predict(X_val3)
val_accuracy3 = accuracy_score(y_val3, y_val_pred3)
print("Validation Accuracy: ", val_accuracy3)

print(classification_report(y_val3, y_val_pred3))

Validation Accuracy:  0.7857142857142857
              precision    recall  f1-score   support

           0       0.88      0.68      0.77        22
           1       0.72      0.90      0.80        20

    accuracy                           0.79        42
   macro avg       0.80      0.79      0.78        42
weighted avg       0.81      0.79      0.78        42



In [9]:
svm_model3 = LinearSVC(random_state=12345)
svm_model3.fit(X_train3, y_train3)
y_pred3 = svm_model3.predict(X_test3)

accuracy3 = accuracy_score(y_test3, y_pred3)
print("Test Accuracy:", accuracy3)

print(classification_report(y_test3, y_pred3))

Test Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.75      0.60      0.67        20
           1       0.69      0.82      0.75        22

    accuracy                           0.71        42
   macro avg       0.72      0.71      0.71        42
weighted avg       0.72      0.71      0.71        42



This model shows quite a high training accuracy but overall lower test set scores across the board. This indicates that too many features may be introducing too much unwanted noise. The function below is a custom forward feature selection function. Forward feature selection is the process of adding in a feature, running the model, measuring the accuracy, and if it improves, we keep the feature. If accuracy worsens we remove the feature and move on to the next one. The goal is to pick a set of optimal features which yield the highest level of accuracy.

In [13]:
# Forward Feature Selection Function

def forwardFeatureSelection():
    bestFeatures = []
    allFeatures = correlation_matrix[last_column].drop(last_column).sort_values(ascending=False).index.tolist()
    highestAccuracy = 0.00
    for feature in allFeatures:
        bestFeatures.append(feature)
        
        X2 = dataset[bestFeatures].values
        y2 = dataset[last_column].values

        X_train2, X_test2, y_train2, y_test2 = train_test_split(
            X2, y2, test_size=0.2, random_state=12345
        )

        X_train2, X_val2, y_train2, y_val2 = train_test_split(
            X_train2, y_train2, test_size=0.25, random_state=12345
        )

        svm_model2 = LinearSVC(random_state=12345)
        svm_model2.fit(X_train2, y_train2)
        y_val_pred2 = svm_model2.predict(X_val2)
        val_accuracy2 = accuracy_score(y_val2, y_val_pred2)

        if val_accuracy2 > highestAccuracy:
            highestAccuracy = val_accuracy2
        else:
            bestFeatures.pop()

    # Running one last time with best features to get test set accuracy
        
    X2 = dataset[bestFeatures].values
    y2 = dataset[last_column].values

    X_train2, X_test2, y_train2, y_test2 = train_test_split(
        X2, y2, test_size=0.2, random_state=12345
    )

    X_train2, X_val2, y_train2, y_val2 = train_test_split(
        X_train2, y_train2, test_size=0.25, random_state=12345
    )

    svm_model2 = LinearSVC(random_state=12345)
    svm_model2.fit(X_train2, y_train2)
    y_pred2 = svm_model2.predict(X_test2)

    accuracy2 = accuracy_score(y_test2, y_pred2)

    print("Best Features:", bestFeatures)
    print("Best Validation Accuracy:", highestAccuracy)
    print("Test Accuracy:", accuracy2)

forwardFeatureSelection()

Best Features: [10, 11, 50, 13, 39]
Best Validation Accuracy: 0.8333333333333334
Test Accuracy: 0.7619047619047619


The forward feature selection model achieved the highest validation accuracy so far with 83% and selected five optimal features out of 60. However the model using the top four correlated features achieved the highest test accuracy so far of 81%. Overall the results suggest that careful feature selection is important for this task, with both too few and too many features leading to lower accuracy. The small dataset remains the biggest limiting factor and a larger dataset would likely yield more consistent and reliable results. The next code block looks at hyperparameter tuning, specifically using GridSearchCV to find the best value for the parameter 'C'. 'C' is a regularization parameter that balances maximizing the margin and minimizing the number of misclassifications in the training data.

In [31]:
from sklearn.model_selection import GridSearchCV 

X3 = dataset[[10, 11, 50, 13, 39]].values
y3 = dataset[last_column].values

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=12345
)

X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_train3, y_train3, test_size=0.25, random_state=12345
)

param_grid = {'C': [0.1, 1, 10, 100, 1000]}

svm_model3 = GridSearchCV(LinearSVC(random_state=12345), param_grid, refit = True, verbose = 3)

svm_model3.fit(X_train3, y_train3)
y_val_pred3 = svm_model3.predict(X_val3)
val_accuracy3 = accuracy_score(y_val3, y_val_pred3)

print("Validation Accuracy: ", val_accuracy3)

print(classification_report(y_val3, y_val_pred3))

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 1/5] END .............................C=0.1;, score=0.760 total time=   0.0s
[CV 2/5] END .............................C=0.1;, score=0.840 total time=   0.0s
[CV 3/5] END .............................C=0.1;, score=0.760 total time=   0.0s
[CV 4/5] END .............................C=0.1;, score=0.920 total time=   0.0s
[CV 5/5] END .............................C=0.1;, score=0.625 total time=   0.0s
[CV 1/5] END ...............................C=1;, score=0.800 total time=   0.0s
[CV 2/5] END ...............................C=1;, score=0.800 total time=   0.0s
[CV 3/5] END ...............................C=1;, score=0.720 total time=   0.0s
[CV 4/5] END ...............................C=1;, score=0.880 total time=   0.0s
[CV 5/5] END ...............................C=1;, score=0.583 total time=   0.0s
[CV 1/5] END ..............................C=10;, score=0.840 total time=   0.0s
[CV 2/5] END ..............................C=10;,

In [32]:
y_pred3 = svm_model3.predict(X_test3)

accuracy3 = accuracy_score(y_test3, y_pred3)
print("Test Accuracy:", accuracy3)

print(classification_report(y_test3, y_pred3))

Test Accuracy: 0.7619047619047619
              precision    recall  f1-score   support

           0       0.92      0.55      0.69        20
           1       0.70      0.95      0.81        22

    accuracy                           0.76        42
   macro avg       0.81      0.75      0.75        42
weighted avg       0.80      0.76      0.75        42



After tuning 'C' the validation accuracy is reduced and the test accuracy remains unchanged. In the final code block below, a non-linear SVM model using SVC() is tuned and trained using the optimal features from the forward feature selection function. An RBF (Radial Basis Function) kernel is used, this defines how data is transformed for separating classes, allowing for the examination of non-linear relationships. Gamma determines how far the influence of each data point reaches in relation to the decision boundary.

In [34]:
from sklearn.model_selection import GridSearchCV 
X3 = dataset[[10, 11, 50, 13, 39]].values
y3 = dataset[last_column].values
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=12345
)
X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_train3, y_train3, test_size=0.25, random_state=12345
)
param_grid = {'C': [0.1, 1, 10, 100, 1000],
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
              'kernel': ['rbf']}
svm_model3 = GridSearchCV(SVC(random_state=12345), param_grid, refit=True, verbose=3)
svm_model3.fit(X_train3, y_train3)
y_val_pred3 = svm_model3.predict(X_val3)
val_accuracy3 = accuracy_score(y_val3, y_val_pred3)
print("Validation Accuracy: ", val_accuracy3)
print(classification_report(y_val3, y_val_pred3))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV 1/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 2/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 3/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 4/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 5/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.542 total time=   0.0s
[CV 1/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 2/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 3/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 4/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.560 total time=   0.0s
[CV 5/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.542 total time=   0.0s
[CV 1/5] END .....C=0.1, gamma=0.01, kernel=rbf;, score=0.560 total time=   0.0s
[CV 2/5] END .....C=0.1, gamma=0.01, kernel=rbf

In [35]:
y_pred3 = svm_model3.predict(X_test3)

accuracy3 = accuracy_score(y_test3, y_pred3)
print("Test Accuracy:", accuracy3)

print(classification_report(y_test3, y_pred3))

Test Accuracy: 0.7857142857142857
              precision    recall  f1-score   support

           0       0.87      0.65      0.74        20
           1       0.74      0.91      0.82        22

    accuracy                           0.79        42
   macro avg       0.80      0.78      0.78        42
weighted avg       0.80      0.79      0.78        42



This is the best result yet. Test accuracy now rises to 79% and validation accuracy curiously is the exact same at 79%. Recall for mines is at 91% which is a 9% improvement on the very first model however, there is a trade off as the recall for rocks actually decreases to 65%. This is okay however, as in a real world scenario missclassifying a rock is far more acceptable than missclassifying a mine.

### Conclusion
A Linear SVM model was used to predict whether a particular sonar response corresponded to a rock or a sea mine. The first part of this notebook looked at feature selection techniques such as manual feature selection and forward feature selection. This proved important as careful feature selection saw improvement in accuracy. However, due to the small dataset size, these improvements were modest. The next section of the notebook looks at hyperparameter tuning for both linear and non-linear SVM models. Tuning the linear model made no difference to accuracy but tuning a new non-linear model resulted in the highest accuracy (79%) and recall (for sea mines: 91%) scores across all the models. This indicates that there are noticable non-linear relationships between different sonar pulses of different energy levels.